# Next-token prediction en WikiText: FFN vs LSTM vs Transformer (DDP 4×GPU con TorchDistributor)

Notebook **simple y modificable**. Incluye:
- Carga + tokenización (Hugging Face)
- Dataset de language modeling por bloques
- 3 modelos (FFN, LSTM, Transformer decoder-only)
- Entrenamiento + validación + test
- **Multi-GPU (4× T4) con TorchDistributor + PyTorch DDP**
- Tracking con MLflow (solo rank0)

Nota: si el dataset/config no existe, hacemos fallback a `wikitext-103-raw-v1`.

## 0) Setup

In [0]:
#Check entorno

import sys, torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CXX11 ABI:", getattr(torch._C, "_GLIBCXX_USE_CXX11_ABI", "n/a"))

import subprocess
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,"]).decode())

Python: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
Torch: 2.0.1+cu118
Torch CUDA: 11.8
CXX11 ABI: False
name, driver_version
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09



In [0]:
dbutils.library.restartPython()

In [0]:
%pip install --no-build-isolation "mamba-ssm[causal-conv1d]==2.2.2"

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import mamba_ssm
from mamba_ssm import Mamba
print("mamba_ssm OK:", getattr(mamba_ssm, "__version__", "unknown"))

2026-05-16 23:02:13.551429: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-16 23:02:13.551480: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-16 23:02:13.551529: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-16 23:02:13.560480: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


mamba_ssm OK: 2.2.2


## 1) Config

In [0]:
import os, math, time, random
from dataclasses import dataclass
from typing import Optional, Dict, List
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer

import mlflow

# --- Mamba LM ---
try:
    from mamba_ssm import Mamba
except Exception as e:
    Mamba = None
    _mamba_import_error = e

# Example values to try: 2, 4, maybe 8 (depends on CPU cores)
os.environ["OMP_NUM_THREADS"] = "1"
print("OMP_NUM_THREADS =", os.environ.get("OMP_NUM_THREADS"), flush=True)

try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    os.environ["DATABRICKS_HOST"] = ctx.apiUrl().get()
    os.environ["DATABRICKS_TOKEN"] = ctx.apiToken().get()
    os.environ["MLFLOW_ENABLE_DB_SDK"] = "true"
except Exception:
    # Si por algún motivo no hay contexto (raro en Jobs), se ignora
    pass

OMP_NUM_THREADS = 1


In [0]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class CFG:
    # =========================
    # Dataset
    # =========================
    DATASET_NAME: str = "wikitext"
    DATASET_CONFIG: str = "wikitext-103-raw-v1"
    CACHE_DIR: str = "/dbfs/cache/hf_wikitext"

    # =========================
    # Longitud de contexto  ← CAMBIAR POR CADA RUN
    # =========================
    BLOCK_SIZE: int = 1024        # {128, 256, 512, 1024}

    # Dataset completo (sin submuestreo)
    TRAIN_FRACTION: Optional[float] = None     # ← era 0.05
    MAX_TRAIN_BLOCKS: Optional[int] = None     # ← era 50_000
    MAX_EVAL_BLOCKS: Optional[int] = 5_000     # eval rápido, suficiente

    # =========================
    # Batch sizing
    # =========================
    TOKENS_PER_GPU_TARGET: int = 8192    # micro-batch (para que quepa en V100 16GB)
    BATCH_SIZE: Optional[int] = None       # se auto-calcula
    GRAD_ACCUM_STEPS: int = 1              # ← FIJO, ya no auto-calculado (1 con 8 gpus, 2 con 4)
    # → tokens/step global = 8192 * 2 * 4 GPUs = 65,536 siempre

    # (EFFECTIVE_BATCH_PER_GPU ya no se usa, puedes dejarlo o quitarlo)

    # =========================
    # Entrenamiento
    # =========================
    SEED: int = 42
    EPOCHS: int = 5
    LR: float = 3e-4
    WEIGHT_DECAY: float = 0.01
    GRAD_CLIP: float = 1.0

    USE_AMP: bool = False                  # ← False: Mamba kernels + Transformer fp16 dan problemas
    USE_GRAD_CHECKPOINTING: bool = True    # necesario para Transformer a 1024

    # =========================
    # Arquitectura (igual para todos)
    # =========================
    D_MODEL: int = 256
    DROPOUT: float = 0.1
    N_HEADS: int = 8
    N_LAYERS: int = 4
    FFN_DIM: int = 1024

    # LSTM
    LSTM_LAYERS: int = 2

    # RoPE
    ROPE_BASE: int = 10000

    # Mamba
    MAMBA_LAYERS: int = 4
    MAMBA_D_STATE: int = 16
    MAMBA_D_CONV: int = 4
    MAMBA_EXPAND: int = 2

    # MoE
    MOE_NUM_EXPERTS: int = 4
    MOE_TOP_K: int = 2
    MOE_AUX_LOSS_COEFF: float = 0.01

    # Jamba
    JAMBA_LAYERS: int = 4
    JAMBA_ATTN_EVERY_N: int = 2    # M → A → M → A

    # =========================
    # MLflow
    # =========================
    EXPERIMENT_NAME: str = "/Shared/next_token_wikitext_comparison_ddp"
    RUN_NAME: str = "full_comparison_bs_" + str(BLOCK_SIZE) + "_" + str(TOKENS_PER_GPU_TARGET) + "_fix" # ← cambiar por BLOCK_SIZE

    # =========================
    # Derivados
    # =========================
    GLOBAL_BATCH_SIZE: int = field(init=False, default=0)
    EFFECTIVE_BATCH_SIZE_PER_GPU: int = field(init=False, default=0)

    def recompute(self) -> None:
        if self.BATCH_SIZE is None:
            self.BATCH_SIZE = max(1, self.TOKENS_PER_GPU_TARGET // self.BLOCK_SIZE)
        self.EFFECTIVE_BATCH_SIZE_PER_GPU = int(self.BATCH_SIZE * self.GRAD_ACCUM_STEPS)

    def summary(self) -> str:
        toks_per_step_per_gpu = self.BATCH_SIZE * self.GRAD_ACCUM_STEPS * self.BLOCK_SIZE
        return (
            f"BLOCK_SIZE={self.BLOCK_SIZE} | micro_batch={self.BATCH_SIZE} | "
            f"grad_accum={self.GRAD_ACCUM_STEPS} | "
            f"tokens/step/GPU={toks_per_step_per_gpu:,} | "
            f"USE_AMP={self.USE_AMP} | EPOCHS={self.EPOCHS}"
        )

# ---- instancia ----
cfg = CFG()
cfg.recompute()
print(cfg.summary())

BLOCK_SIZE=1024 | micro_batch=8 | grad_accum=1 | tokens/step/GPU=8,192 | USE_AMP=False | EPOCHS=5


In [0]:
# Sanity check: GPUs disponibles
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

CUDA available: True
GPU count: 8


## 2) Dataset + tokenización

In [0]:
# Tokenizer (GPT-2)
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
vocab_size = tokenizer.vocab_size
print("Vocab:", vocab_size)

warnings.filterwarnings(
    "ignore",
    message=r'During large dataset downloads*',
    category=UserWarning,
)

def load_wikitext_with_fallback(name: str, config_name: str, cache_dir: str):
    try:
        ds = load_dataset(name, config_name, cache_dir=cache_dir)
        print(f"Loaded: {name}/{config_name}")
        return ds, config_name
    except Exception as e:
        print(f"[WARN] No pude cargar {name}/{config_name} -> {e}")
        fallback = "wikitext-103-raw-v1"
        ds = load_dataset(name, fallback, cache_dir=cache_dir)
        print(f"Loaded fallback: {name}/{fallback}")
        return ds, fallback

raw_ds, used_config = load_wikitext_with_fallback(cfg.DATASET_NAME, cfg.DATASET_CONFIG, cfg.CACHE_DIR)


def tokenize_fn(batch):
    texts = [t for t in batch["text"] if t and not t.isspace()]
    return tokenizer(texts, return_attention_mask=False)


def group_texts(examples, block_size: int):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_len = len(concatenated["input_ids"])
    usable_len = (total_len // (block_size + 1)) * (block_size + 1)
    if usable_len == 0:
        return {"input_ids": [], "labels": []}

    ids = concatenated["input_ids"][:usable_len]
    input_blocks, label_blocks = [], []
    step = block_size + 1
    for i in range(0, usable_len, step):
        chunk = ids[i:i+step]
        input_blocks.append(chunk[:-1])
        label_blocks.append(chunk[1:])
    return {"input_ids": input_blocks, "labels": label_blocks}


def prepare_split(ds_split, split_name: str, max_blocks: Optional[int]):

    # Opcional: submuestreo rápido
    if cfg.TRAIN_FRACTION is not None and split_name == "train":
        ds_split = ds_split.shuffle(seed=cfg.SEED)
        take_n = max(1, int(len(ds_split) * cfg.TRAIN_FRACTION))
        ds_split = ds_split.select(range(take_n))
        
    tokenized = ds_split.map(tokenize_fn, batched=True, remove_columns=ds_split.column_names, desc=f"tok {split_name}")
    lm_ds = tokenized.map(lambda x: group_texts(x, cfg.BLOCK_SIZE), batched=True, desc=f"group {split_name}")
    if max_blocks is not None and len(lm_ds) > max_blocks:
        lm_ds = lm_ds.select(range(max_blocks))
    lm_ds.set_format(type="torch", columns=["input_ids", "labels"])
    return lm_ds

train_ds = prepare_split(raw_ds["train"], "train", cfg.MAX_TRAIN_BLOCKS)
valid_ds = prepare_split(raw_ds["validation"], "validation", cfg.MAX_EVAL_BLOCKS)
test_ds  = prepare_split(raw_ds["test"], "test", cfg.MAX_EVAL_BLOCKS)

print("Blocks:", len(train_ds), len(valid_ds), len(test_ds))

/databricks/python/lib/python3.10/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Vocab: 50257
Loaded: wikitext/wikitext-103-raw-v1


group train:   0%|          | 0/1165029 [00:00<?, ? examples/s]

group validation:   0%|          | 0/2461 [00:00<?, ? examples/s]

group test:   0%|          | 0/2891 [00:00<?, ? examples/s]

Blocks: 114471 239 274


## 3) Modelos

In [0]:
class FFNBaselineLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, dropout: float):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_model, vocab_size)
        )

    def forward(self, input_ids):
        x = self.emb(input_ids)             # (B,T,D)
        cumsum = x.cumsum(dim=1)
        denom = torch.arange(1, x.size(1)+1, device=x.device).view(1, -1, 1)
        ctx = cumsum / denom
        return self.mlp(ctx)                # (B,T,V)


class LSTMLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_layers: int, dropout: float):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.lstm = nn.LSTM(d_model, d_model, num_layers=n_layers, dropout=dropout if n_layers>1 else 0.0, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        x = self.emb(input_ids)
        out, _ = self.lstm(x)
        out = self.drop(out)
        return self.head(out)



class VanillaCausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        att = att.masked_fill(causal, float("-inf"))

        w = torch.softmax(att, dim=-1)
        w = F.dropout(w, p=self.dropout, training=self.training)

        y = w @ v
        y = y.transpose(1, 2).contiguous().view(B, T, D)
        return self.out(y)


class VanillaTransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.att = VanillaCausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class TransformerDecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_heads: int,
                 n_layers: int, ffn_dim: int, dropout: float, max_len: int):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            VanillaTransformerBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        B, T = input_ids.shape
        pos_ids = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos_ids)
        x = self.drop(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.ln_f(x)
        return self.head(x)

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RotaryEmbedding(nn.Module):
    """
    Precompute cos/sin tables for RoPE up to max_seq_len.
    """
    def __init__(self, head_dim: int, max_seq_len: int, base: int = 10000):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE necesita head_dim par."
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        self.base = base

        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))  # (head_dim/2,)
        self.register_buffer("inv_freq", inv_freq, persistent=False)

        # caches (se inicializan a None y se crean en el primer forward)
        self.register_buffer("_cos", None, persistent=False)
        self.register_buffer("_sin", None, persistent=False)

    def _build_cache(self, device, dtype):
        t = torch.arange(self.max_seq_len, device=device, dtype=torch.float32)  # siempre float32 para trig estable
        freqs = torch.einsum("t,f->tf", t, self.inv_freq.to(device=device))    # (T, head_dim/2)
        cos = freqs.cos().to(dtype=dtype)
        sin = freqs.sin().to(dtype=dtype)
        # shape para broadcasting sobre (B, H, T, head_dim/2)
        self._cos = cos[None, None, :, :]   # (1,1,T,hd/2)
        self._sin = sin[None, None, :, :]   # (1,1,T,hd/2)

    def forward(self, seq_len: int, device, dtype):
        if self._cos is None or self._sin is None or self._cos.device != device or self._cos.dtype != dtype:
            self._build_cache(device, dtype)
        return self._cos[:, :, :seq_len, :], self._sin[:, :, :seq_len, :]


def apply_rope(x, cos, sin):
    """
    x: (B, H, T, head_dim)
    cos/sin: (1, 1, T, head_dim/2)
    """
    x1 = x[..., 0::2]  # (B,H,T,hd/2)
    x2 = x[..., 1::2]  # (B,H,T,hd/2)
    xr1 = x1 * cos - x2 * sin
    xr2 = x1 * sin + x2 * cos
    # interleave back
    out = torch.stack((xr1, xr2), dim=-1).flatten(-2)  # (B,H,T,hd)
    return out


# -------------------------
# RoPE Causal Self-Attention
# -------------------------
class RoPECausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float, max_seq_len: int, rope_base: int = 10000):
        super().__init__()
        assert d_model % n_heads == 0
        head_dim = d_model // n_heads
        assert head_dim % 2 == 0, "Para RoPE, (d_model/n_heads) debe ser par."

        self.n_heads = n_heads
        self.head_dim = head_dim
        self.dropout = dropout

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

        self.rope = RotaryEmbedding(head_dim=head_dim, max_seq_len=max_seq_len, base=rope_base)

    def forward(self, x):
        """
        x: (B, T, D)
        return: (B, T, D)
        """
        B, T, D = x.shape
        qkv = self.qkv(x)                         # (B, T, 3D)
        q, k, v = qkv.chunk(3, dim=-1)            # each (B, T, D)

        # -> (B, H, T, hd)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # RoPE en q,k
        cos, sin = self.rope(seq_len=T, device=x.device, dtype=q.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # scaled dot-product attention + causal mask
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)   # (B,H,T,T)
        causal = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        att = att.masked_fill(causal, float("-inf"))

        w = torch.softmax(att, dim=-1)                                # (B,H,T,T)
        w = F.dropout(w, p=self.dropout, training=self.training)

        y = w @ v                                                     # (B,H,T,hd)
        y = y.transpose(1, 2).contiguous().view(B, T, D)              # (B,T,D)
        return self.out(y)


# -------------------------
# Transformer RoPE Block
# -------------------------
class TransformerRoPEBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float, max_seq_len: int, rope_base: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.att = RoPECausalSelfAttention(d_model, n_heads, dropout, max_seq_len=max_seq_len, rope_base=rope_base)
        self.ln2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


# -------------------------
# Decoder-only LM with RoPE
# -------------------------
class TransformerRoPELM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_heads: int, n_layers: int,
                 ffn_dim: int, dropout: float, max_seq_len: int, rope_base: int = 10000):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerRoPEBlock(d_model, n_heads, ffn_dim, dropout, max_seq_len=max_seq_len, rope_base=rope_base)
            for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        """
        input_ids: (B, T) -> logits (B, T, vocab)
        """
        x = self.tok_emb(input_ids)           # (B,T,D)
        x = self.drop(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits


class MambaLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, d_state, d_conv, expand, dropout):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        # Cada bloque: LayerNorm + Mamba (con residual en forward)
        self.blocks = nn.ModuleList([
            Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        x = self.emb(input_ids)
        x = self.drop(x)
        for norm, blk in zip(self.norms, self.blocks):
            x = x + blk(norm(x))       # ← pre-norm + residual
        x = self.norm(x)
        logits = self.head(x)
        return logits


# --- ALiBi helpers ---
import math

def _alibi_slopes(n_heads: int) -> torch.Tensor:
    """
    Slopes típicos de ALiBi: serie geométrica por cabeza.
    Implementación estándar usada en múltiples refs públicas (ver repos ALiBi). [1](https://github.com/jaketae/alibi)
    """
    def _get_slopes_power_of_2(n):
        start = 2.0 ** (-8.0 / n)
        return torch.pow(start, torch.arange(1, n + 1))

    if math.log2(n_heads).is_integer():
        return _get_slopes_power_of_2(n_heads)

    n = 2 ** math.floor(math.log2(n_heads))
    slopes = _get_slopes_power_of_2(n)
    extra = _get_slopes_power_of_2(2 * n)[0::2][: (n_heads - n)]
    return torch.cat([slopes, extra], dim=0)


class ALiBiSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

        # slopes fijos (no entrenables)
        self.register_buffer("slopes", _alibi_slopes(n_heads).view(1, n_heads, 1, 1), persistent=False)

    def forward(self, x: torch.Tensor):
        # x: (B,T,D)
        B, T, D = x.shape
        qkv = self.qkv(x)  # (B,T,3D)
        q, k, v = qkv.chunk(3, dim=-1)

        # (B, heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # scores: (B, heads, T, T)
        scores = (q @ k.transpose(-2, -1)) * self.scale

        # causal mask
        causal = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(causal, float("-inf"))

        # ALiBi bias: -|i-j| (en causal basta i-j para j<=i)
        i = torch.arange(T, device=x.device)
        j = torch.arange(T, device=x.device)
        dist = (i[:, None] - j[None, :]).clamp(min=0)  # (T,T)
        alibi = -dist.view(1, 1, T, T) * self.slopes.to(x.device)  # (1,heads,T,T)
        scores = scores + alibi

        attn = torch.softmax(scores, dim=-1)
        attn = self.drop(attn)

        out = attn @ v  # (B,heads,T,head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.proj(out)


class TransformerALiBiLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_heads: int, n_layers: int, ffn_dim: int, dropout: float):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)

        self.layers = nn.ModuleList([])
        for _ in range(n_layers):
            attn = ALiBiSelfAttention(d_model=d_model, n_heads=n_heads, dropout=dropout)
            ff = nn.Sequential(
                nn.Linear(d_model, ffn_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(ffn_dim, d_model),
                nn.Dropout(dropout),
            )
            ln1 = nn.LayerNorm(d_model)
            ln2 = nn.LayerNorm(d_model)
            self.layers.append(nn.ModuleDict({"attn": attn, "ff": ff, "ln1": ln1, "ln2": ln2}))
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids: torch.Tensor):
        x = self.emb(input_ids)   # (B,T,D)
        x = self.drop(x)

        for layer in self.layers:
            x = x + layer["attn"](layer["ln1"](x))
            x = x + layer["ff"](layer["ln2"](x))
        x = self.ln_f(x)
        logits = self.head(x)     # (B,T,V)
        return logits
    
class MoERouter(nn.Module):
    """
    Router top-k para MoE.
    Para cada token, calcula scores sobre E expertos y selecciona los top_k.
    Devuelve pesos normalizados, índices y una pérdida auxiliar de balanceo.
    """
    def __init__(self, d_model: int, num_experts: int, top_k: int):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x: torch.Tensor):
        """
        x: (B, T, D)
        returns:
            weights:  (B, T, top_k)  — pesos normalizados de los expertos elegidos
            indices:  (B, T, top_k)  — índices de los expertos elegidos
            aux_loss: escalar        — pérdida de balanceo de carga
        """
        logits = self.gate(x)                                  # (B, T, E)
        probs  = torch.softmax(logits, dim=-1)                 # (B, T, E)

        top_k_w, top_k_idx = torch.topk(probs, self.top_k, dim=-1)
        top_k_w = top_k_w / (top_k_w.sum(dim=-1, keepdim=True) + 1e-9)

        # Load balancing loss (Switch Transformer, Fedus et al. 2022)
        # L_aux = E * sum_i(f_i * p_i)
        # f_i = fracción de tokens asignados al experto i (top-1)
        # p_i = probabilidad media del router para el experto i
        with torch.no_grad():
            top1 = top_k_idx[..., 0]                           # (B, T)
            f = F.one_hot(top1, self.num_experts).float().mean(dim=(0, 1))  # (E,)
        p = probs.mean(dim=(0, 1))                             # (E,)
        aux_loss = self.num_experts * (f * p).sum()

        return top_k_w, top_k_idx, aux_loss


class MoEFFN(nn.Module):
    """
    Capa Mixture of Experts: E expertos FFN independientes con routing sparse.
    Implementación token-level iterando sobre expertos (eficiente para E pequeño).
    """
    def __init__(self, d_model: int, ffn_dim: int, num_experts: int, top_k: int, dropout: float):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.router = MoERouter(d_model, num_experts, top_k)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, ffn_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(ffn_dim, d_model),
                nn.Dropout(dropout),
            )
            for _ in range(num_experts)
        ])

    def forward(self, x: torch.Tensor):
        """
        x: (B, T, D)
        returns: (output (B,T,D), aux_loss escalar)
        """
        B, T, D = x.shape
        weights, indices, aux_loss = self.router(x)

        x_flat = x.view(B * T, D)
        w_flat = weights.view(B * T, self.top_k)
        i_flat = indices.view(B * T, self.top_k)

        out = torch.zeros_like(x_flat)

        for e in range(self.num_experts):
            mask = (i_flat == e)                        # (N, k) bool
            if not mask.any():
                continue
            token_mask = mask.any(dim=-1)               # (N,)
            tok_ids = token_mask.nonzero(as_tuple=True)[0]

            expert_out = self.experts[e](x_flat[tok_ids])
            w = (w_flat[tok_ids] * mask[tok_ids].float()).sum(dim=-1, keepdim=True)
            out[tok_ids] += w * expert_out

        return out.view(B, T, D), aux_loss


# =========================================================
# Transformer RoPE + MoE  (bloque y modelo LM)
# =========================================================

class TransformerRoPE_MoE_Block(nn.Module):
    """Pre-norm block: LN→RoPE Attn + residual, LN→MoE FFN + residual."""
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float,
                 max_seq_len: int, rope_base: int, num_experts: int, top_k: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.att = RoPECausalSelfAttention(d_model, n_heads, dropout,
                                           max_seq_len=max_seq_len, rope_base=rope_base)
        self.ln2 = nn.LayerNorm(d_model)
        self.moe = MoEFFN(d_model, ffn_dim, num_experts, top_k, dropout)

    def forward(self, x):
        """returns: (x (B,T,D), aux_loss escalar)"""
        x = x + self.att(self.ln1(x))
        moe_out, aux_loss = self.moe(self.ln2(x))
        x = x + moe_out
        return x, aux_loss


class TransformerRoPE_MoE_LM(nn.Module):
    """
    Transformer decoder-only con RoPE + Mixture of Experts.
    Tras el forward, self._aux_loss contiene la pérdida auxiliar acumulada
    (media sobre capas), lista para sumar a la loss principal.
    """
    def __init__(self, vocab_size: int, d_model: int, n_heads: int, n_layers: int,
                 ffn_dim: int, dropout: float, max_seq_len: int, rope_base: int = 10000,
                 num_experts: int = 4, top_k: int = 2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerRoPE_MoE_Block(
                d_model, n_heads, ffn_dim, dropout,
                max_seq_len=max_seq_len, rope_base=rope_base,
                num_experts=num_experts, top_k=top_k,
            )
            for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self._aux_loss = 0.0

    def forward(self, input_ids):
        x = self.tok_emb(input_ids)
        x = self.drop(x)

        total_aux = 0.0
        for blk in self.blocks:
            x, aux = blk(x)
            total_aux = total_aux + aux

        self._aux_loss = total_aux / len(self.blocks)

        x = self.ln_f(x)
        return self.lm_head(x)


# =========================================================
# Jamba — Híbrido Mamba + Attention (estilo AI21 Labs)
# =========================================================

class JambaBlock(nn.Module):
    """
    Bloque unificado que puede ser Mamba o Attention (RoPE), ambos pre-norm + residual.
    """
    def __init__(self, block_type: str, d_model: int, n_heads: int, ffn_dim: int,
                 dropout: float, max_seq_len: int, rope_base: int,
                 mamba_d_state: int, mamba_d_conv: int, mamba_expand: int):
        super().__init__()
        self.block_type = block_type

        if block_type == "attention":
            self.ln1 = nn.LayerNorm(d_model)
            self.att = RoPECausalSelfAttention(d_model, n_heads, dropout,
                                               max_seq_len=max_seq_len, rope_base=rope_base)
            self.ln2 = nn.LayerNorm(d_model)
            self.ff = nn.Sequential(
                nn.Linear(d_model, ffn_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(ffn_dim, d_model),
                nn.Dropout(dropout),
            )
        elif block_type == "mamba":
            self.ln = nn.LayerNorm(d_model)
            self.mamba = Mamba(d_model=d_model, d_state=mamba_d_state,
                              d_conv=mamba_d_conv, expand=mamba_expand)
        else:
            raise ValueError(f"block_type debe ser 'mamba' o 'attention', got '{block_type}'")

    def forward(self, x):
        if self.block_type == "attention":
            x = x + self.att(self.ln1(x))
            x = x + self.ff(self.ln2(x))
        else:
            x = x + self.mamba(self.ln(x))
        return x


class JambaLM(nn.Module):
    """
    Jamba: híbrido Mamba + Attention.

    Con n_layers=4 y attn_every_n=2 (default):
      capa 0 → Mamba,  capa 1 → Attention,  capa 2 → Mamba,  capa 3 → Attention
      Layout: M → A → M → A

    Con attn_every_n=3:
      capa 0 → Mamba,  capa 1 → Mamba,  capa 2 → Attention,  capa 3 → Mamba
      Layout: M → M → A → M
    """
    def __init__(self, vocab_size: int, d_model: int, n_heads: int, n_layers: int,
                 ffn_dim: int, dropout: float, max_seq_len: int, rope_base: int = 10000,
                 mamba_d_state: int = 16, mamba_d_conv: int = 4, mamba_expand: int = 2,
                 attn_every_n: int = 2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList()
        for i in range(n_layers):
            btype = "attention" if ((i + 1) % attn_every_n == 0) else "mamba"
            self.blocks.append(JambaBlock(
                block_type=btype, d_model=d_model, n_heads=n_heads,
                ffn_dim=ffn_dim, dropout=dropout,
                max_seq_len=max_seq_len, rope_base=rope_base,
                mamba_d_state=mamba_d_state, mamba_d_conv=mamba_d_conv,
                mamba_expand=mamba_expand,
            ))

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        x = self.tok_emb(input_ids)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        return self.lm_head(x)

    def block_layout(self) -> str:
        """Devuelve string del layout, ej: 'M → A → M → A'."""
        return " → ".join("A" if b.block_type == "attention" else "M" for b in self.blocks)



## 4) Loss

In [0]:
loss_fn = nn.CrossEntropyLoss()

def compute_loss(logits, labels):
    B, T, V = logits.shape
    return loss_fn(logits.view(B*T, V), labels.view(B*T))

## 5) DDP + TorchDistributor

In [0]:
from pyspark.ml.torch.distributor import TorchDistributor

import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from datetime import timedelta

def ddp_setup():
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)

    dist.init_process_group("nccl", timeout=timedelta(minutes=3))

    dist.barrier(device_ids=[local_rank])
    return local_rank

def ddp_barrier(local_rank: int):
    dist.barrier(device_ids=[local_rank])


def ddp_cleanup():
    dist.destroy_process_group()


def is_main_process():
    return (not dist.is_available()) or (not dist.is_initialized()) or dist.get_rank() == 0


def collate(batch):
    input_ids = torch.stack([x["input_ids"] for x in batch])
    labels = torch.stack([x["labels"] for x in batch])
    return {"input_ids": input_ids, "labels": labels}


def make_dataloaders_ddp(rank: int, world_size: int):
    train_sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True, drop_last=True)
    val_sampler   = DistributedSampler(valid_ds, num_replicas=world_size, rank=rank, shuffle=False, drop_last=False)
    test_sampler  = DistributedSampler(test_ds,  num_replicas=world_size, rank=rank, shuffle=False, drop_last=False)

    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, sampler=train_sampler, collate_fn=collate, drop_last=True)
    valid_loader = DataLoader(valid_ds, batch_size=cfg.BATCH_SIZE, sampler=val_sampler, collate_fn=collate, drop_last=False)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, sampler=test_sampler, collate_fn=collate, drop_last=False)
    return train_loader, valid_loader, test_loader, train_sampler


@torch.no_grad()
def evaluate_ddp(model, loader):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(torch.cuda.current_device(), non_blocking=True)
        labels = batch["labels"].to(torch.cuda.current_device(), non_blocking=True)
        logits = model(input_ids)
        loss = compute_loss(logits, labels)

        B, T = labels.shape
        n_tok = B * T
        total_loss += loss.item() * n_tok
        total_tokens += n_tok

    tens = torch.tensor([total_loss, total_tokens], device=torch.cuda.current_device(), dtype=torch.float64)
    dist.all_reduce(tens, op=dist.ReduceOp.SUM)

    loss_avg = (tens[0] / tens[1]).item()
    ppl = float(math.exp(loss_avg)) if loss_avg < 50 else float("inf")
    return {"loss": float(loss_avg), "ppl": float(ppl)}


from tqdm.auto import tqdm

def train_one_model_ddp(
    model_name: str,
    model: nn.Module,
    train_loader,
    valid_loader,
    test_loader,
    train_sampler,
    local_rank: int,
):
    device = torch.device(f"cuda:{local_rank}")
    model = model.to(device)
    ddp_model = DDP(model, device_ids=[local_rank])

    world_size = dist.get_world_size()
    tokens_seen = 0

    opt = torch.optim.AdamW(ddp_model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

    log_every = 500

    # ← NUEVO: reset del pico de memoria antes de entrenar
    torch.cuda.reset_peak_memory_stats(device)

    for epoch in range(1, cfg.EPOCHS + 1):

        t_epoch = time.time()
        t_last = time.time()
        tokens_since_last_log = 0          # ← NUEVO

        ddp_model.train()
        train_sampler.set_epoch(epoch)

        iterator = train_loader
        if is_main_process():
            iterator = tqdm(train_loader, desc=f"[{model_name}] epoch {epoch}/{cfg.EPOCHS}", leave=False)

        running = 0.0
        steps = 0

        for step, batch in enumerate(iterator, start=1):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            B, T = labels.shape
            step_tokens = B * T * world_size
            tokens_seen += step_tokens
            tokens_since_last_log += step_tokens              # ← NUEVO
            tokens_seen_m = int(tokens_seen // 1_000_000)

            opt.zero_grad(set_to_none=True)
            logits = ddp_model(input_ids)
            loss = compute_loss(logits, labels)

            # --- MoE auxiliary loss ---
            raw_model = ddp_model.module if hasattr(ddp_model, "module") else ddp_model
            if hasattr(raw_model, "_aux_loss") and isinstance(raw_model._aux_loss, torch.Tensor):
                loss = loss + cfg.MOE_AUX_LOSS_COEFF * raw_model._aux_loss

            loss.backward()

            if cfg.GRAD_CLIP is not None:
                nn.utils.clip_grad_norm_(ddp_model.parameters(), cfg.GRAD_CLIP)

            opt.step()

            running += loss.item()
            steps += 1

            if step % log_every == 0 and is_main_process():
                t_now = time.time()
                elapsed = t_now - t_last

                # ms por step
                step_ms = elapsed * 1000.0 / log_every
                mlflow.log_metric("ms_per_step", step_ms, step=tokens_seen_m)

                # ← NUEVO: tokens/s (todos los GPUs combinados)
                if elapsed > 0:
                    mlflow.log_metric("tokens_per_sec", tokens_since_last_log / elapsed, step=tokens_seen_m)

                # ← NUEVO: pico de memoria GPU (rank 0, en MB)
                peak_mb = torch.cuda.max_memory_allocated(device) / (1024 * 1024)
                mlflow.log_metric("peak_gpu_mb", peak_mb, step=tokens_seen_m)

                # train loss
                avg_loss = (running / steps) if steps > 0 else float("nan")
                iterator.set_postfix({"train_loss": f"{avg_loss:.4f}"})
                mlflow.log_metric("train_loss_tokens", float(avg_loss), step=tokens_seen_m)
                mlflow.log_metric("train_tokens_seen_m", tokens_seen / 1_000_000, step=tokens_seen_m)

                t_last = t_now
                tokens_since_last_log = 0                     # ← NUEVO: reset

        # Validación global
        val = evaluate_ddp(ddp_model, valid_loader)

        if is_main_process():
            print(f"[{model_name}] epoch={epoch} val_loss={val['loss']:.4f} val_ppl={val['ppl']:.2f}")
            mlflow.log_metric("val_loss_tokens", val["loss"], step=tokens_seen_m)
            mlflow.log_metric("val_ppl_tokens",  val["ppl"],  step=tokens_seen_m)
            mlflow.log_metric("val_loss", val["loss"], step=epoch)
            mlflow.log_metric("val_ppl",  val["ppl"],  step=epoch)

    # Test global
    test_m = evaluate_ddp(ddp_model, test_loader)

    # ← NUEVO: log final de pico de memoria y throughput medio
    if is_main_process():
        final_peak_mb = torch.cuda.max_memory_allocated(device) / (1024 * 1024)
        mlflow.log_metric("final_peak_gpu_mb", final_peak_mb)

        total_time = time.time() - t_epoch  # última epoch, pero train_seconds ya cubre total
        mlflow.log_metric("final_tokens_per_sec", tokens_seen / (time.time() - t_epoch) if tokens_seen > 0 else 0)

    return test_m, ddp_model

import traceback

def ddp_worker():
    import time, traceback, warnings
    import torch
    import torch.distributed as dist
    import mlflow
    from contextlib import nullcontext

    local_rank = ddp_setup()
    rank = dist.get_rank()
    world_size = dist.get_world_size()

    # --- warning pydantic fuera ---
    warnings.filterwarnings(
        "ignore",
        message=r'Field "model_name" has conflict with protected namespace "model_"*',
        category=UserWarning,
    )

    # Dataloaders
    train_loader, valid_loader, test_loader, train_sampler = make_dataloaders_ddp(rank, world_size)

    # MLflow solo rank0
    if is_main_process():
        mlflow.set_experiment(cfg.EXPERIMENT_NAME)

        # Limpieza por si quedó un run activo en este intérprete
        if mlflow.active_run() is not None:
            mlflow.end_run()  # cierra el run "UNFINISHED" anterior [1](https://docs.ray.io/en/latest/train/huggingface-accelerate.html)[2](https://aaltorse.github.io/simple-huggingface/accelerate/)

    # ✅ Import Mamba aquí, y decide en TODOS los ranks si está disponible
    mamba_ok = True
    try:
        from mamba_ssm import Mamba  # noqa: F401
    except Exception as e:
        mamba_ok = False
        if is_main_process():
            print(f"[WARN] Mamba no disponible, se omite. Error import: {e}")

    # Construye model_builders de forma idéntica en todos los ranks
    model_builders = {
        "FFN": lambda: FFNBaselineLM(vocab_size=vocab_size, d_model=cfg.D_MODEL, dropout=cfg.DROPOUT),
        "LSTM": lambda: LSTMLM(vocab_size=vocab_size, d_model=cfg.D_MODEL, n_layers=cfg.LSTM_LAYERS, dropout=cfg.DROPOUT),
        "Transformer": lambda: TransformerDecoderOnlyLM(
            vocab_size=vocab_size, d_model=cfg.D_MODEL, n_heads=cfg.N_HEADS,
            n_layers=cfg.N_LAYERS, ffn_dim=cfg.FFN_DIM, dropout=cfg.DROPOUT, max_len=cfg.BLOCK_SIZE
        ),
        "Transformer_ALiBi": lambda: TransformerALiBiLM(
            vocab_size=vocab_size, d_model=cfg.D_MODEL, n_heads=cfg.N_HEADS,
            n_layers=cfg.N_LAYERS, ffn_dim=cfg.FFN_DIM, dropout=cfg.DROPOUT
        ),
        "Transformer_RoPE": lambda: TransformerRoPELM(
            vocab_size=vocab_size,
            d_model=cfg.D_MODEL,
            n_heads=cfg.N_HEADS,
            n_layers=cfg.N_LAYERS,
            ffn_dim=cfg.FFN_DIM,
            dropout=cfg.DROPOUT,
            max_seq_len=cfg.BLOCK_SIZE,
            rope_base=getattr(cfg, "ROPE_BASE", 10000),
        ),
        "Transformer_RoPE_MoE": lambda: TransformerRoPE_MoE_LM(
            vocab_size=vocab_size,
            d_model=cfg.D_MODEL,
            n_heads=cfg.N_HEADS,
            n_layers=cfg.N_LAYERS,
            ffn_dim=cfg.FFN_DIM,
            dropout=cfg.DROPOUT,
            max_seq_len=cfg.BLOCK_SIZE,
            rope_base=getattr(cfg, "ROPE_BASE", 10000),
            num_experts=cfg.MOE_NUM_EXPERTS,
            top_k=cfg.MOE_TOP_K,
        ),
    }

    if mamba_ok:
        model_builders["Mamba"] = lambda: MambaLM(
            vocab_size=vocab_size, d_model=cfg.D_MODEL,
            n_layers=cfg.MAMBA_LAYERS, d_state=cfg.MAMBA_D_STATE,
            d_conv=cfg.MAMBA_D_CONV, expand=cfg.MAMBA_EXPAND,
            dropout=cfg.DROPOUT
        )
        model_builders["Jamba"] = lambda: JambaLM(
            vocab_size=vocab_size,
            d_model=cfg.D_MODEL,
            n_heads=cfg.N_HEADS,
            n_layers=cfg.JAMBA_LAYERS,
            ffn_dim=cfg.FFN_DIM,
            dropout=cfg.DROPOUT,
            max_seq_len=cfg.BLOCK_SIZE,
            rope_base=getattr(cfg, "ROPE_BASE", 10000),
            mamba_d_state=cfg.MAMBA_D_STATE,
            mamba_d_conv=cfg.MAMBA_D_CONV,
            mamba_expand=cfg.MAMBA_EXPAND,
            attn_every_n=cfg.JAMBA_ATTN_EVERY_N,
        )

    # ✅ Context manager de MLflow:
    # - rank0: abre run padre
    # - otros ranks: no-op
    parent_ctx = (mlflow.start_run(run_name=cfg.RUN_NAME) if is_main_process() else nullcontext())

    try:
        with parent_ctx:
            if is_main_process():
                # log de config solo una vez
                mlflow.log_params({**cfg.__dict__, "used_config": used_config, "world_size": world_size})
                print(f"[rank0] Starting models: {list(model_builders.keys())}")

            # 🔒 Sync inicial (todos los ranks)
            ddp_barrier(local_rank)

            for name, builder in model_builders.items():
                # 🔒 Sync antes del modelo
                ddp_barrier(local_rank)

                if is_main_process():
                    print(f"\n=== Training model: {name} ===")

                # Nested run SOLO en rank0; en otros ranks es no-op.
                nested_ctx = (mlflow.start_run(run_name=name, nested=True) if is_main_process() else nullcontext())

                try:
                    with nested_ctx:
                        # Construcción del modelo en TODOS los ranks
                        model = builder()

                        if is_main_process():
                            n_params = sum(p.numel() for p in model.parameters())
                            n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
                            print(f"[{name}] params total={n_params:,} trainable={n_train:,}")

                            # params útiles para comparar
                            mlflow.log_param("model_type", name)
                            mlflow.log_param("block_size", cfg.BLOCK_SIZE)
                            mlflow.log_param("micro_batch_per_gpu", cfg.BATCH_SIZE)
                            mlflow.log_param("grad_accum_steps", cfg.GRAD_ACCUM_STEPS)
                            mlflow.log_param("world_size", world_size)
                            mlflow.log_param("n_params_total", int(n_params))
                            mlflow.log_param("n_params_trainable", int(n_train))

                            # Log extra para modelos nuevos
                            if hasattr(model, "block_layout"):
                                layout = model.block_layout()
                                mlflow.log_param("jamba_layout", layout)
                                print(f"[{name}] block layout: {layout}")

                            if hasattr(model, "_aux_loss"):
                                mlflow.log_param("moe_num_experts", cfg.MOE_NUM_EXPERTS)
                                mlflow.log_param("moe_top_k", cfg.MOE_TOP_K)
                                mlflow.log_param("moe_aux_coeff", cfg.MOE_AUX_LOSS_COEFF)

                            print(f"[{name}] entering train_one_model_ddp...")

                        start = time.time()
                        test_m, ddp_model = train_one_model_ddp(
                            name, model,
                            train_loader, valid_loader, test_loader, train_sampler,
                            local_rank=local_rank
                        )
                        elapsed = time.time() - start

                        if is_main_process():
                            print(f"[{name}] finished train_one_model_ddp. Logging test metrics...")

                            mlflow.log_metric("test_loss", float(test_m["loss"]))
                            mlflow.log_metric("test_ppl",  float(test_m["ppl"]))
                            mlflow.log_metric("train_seconds", float(elapsed))

                            model_path = f"model_{name}.pt"
                            torch.save(ddp_model.module.state_dict(), model_path)
                            mlflow.log_artifact(model_path)

                            print(f"=== Done: {name} | test_loss={test_m['loss']:.4f} test_ppl={test_m['ppl']:.2f} ===")

                except Exception as e:
                    # IMPORTANTE: si un rank falla, lo imprimimos en ese rank y relanzamos
                    # Esto evita deadlocks "silenciosos".
                    print(f"[rank{rank}] ERROR in model {name}: {e}")
                    print(traceback.format_exc())
                    raise
                finally:
                    # Limpieza ligera por modelo
                    try:
                        torch.cuda.empty_cache()
                    except Exception:
                        pass

                # 🔒 Sync después del modelo (si todo fue bien hasta aquí)
                ddp_barrier(local_rank)

    finally:
        # Evitamos un barrier "final" aquí porque si algún rank murió, puede colgar.
        # Hacemos cleanup siempre.
        try:
            ddp_cleanup()
        except Exception:
            pass

    return None

## 6) Lanzar en 8 GPUs y mostrar resultados (sin crash con DF vacíos)

In [0]:
_ = TorchDistributor(num_processes=8, local_mode=True, use_gpu=True).run(ddp_worker)
print("Entrenamiento distribuido finalizado. Revisa métricas/artefactos en MLflow.")